# Pharmacy Access in Northern Vermont

Driving time analysis for pharmacies in **Caledonia**, **Orleans**, **Lamoille**, and **Washington** counties.

This notebook:
1. Loads pharmacy locations (NPPES NPI-sourced, pre-geocoded)
2. Clusters nearby pharmacies into centroids to reduce API calls
3. Downloads county boundaries from US Census TIGER/Line
4. Loads address sample points from CSV files
5. Computes driving time to the nearest pharmacy via the **Google Routes API**
6. Visualizes results as an interactive heat map
7. Reports descriptive statistics on pharmacy access

**Budget**: Google Routes API Compute Route Matrix Essentials — 10,000 free elements/month, then $5/1,000. Hard cap: **12,000 elements ($10 max spend)**.

In [1]:
import math
import os
import time
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(Path("../.env"))

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from shapely.geometry import Point

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

ADDRESSES_DIR = DATA_DIR / "addresses"

TARGET_COUNTIES = {
    "Caledonia": "50005",
    "Orleans": "50019",
    "Lamoille": "50015",
    "Washington": "50023",
}
COUNTY_FIPS_TO_NAME = {v: k for k, v in TARGET_COUNTIES.items()}

GOOGLE_API_KEY = os.environ.get("GOOGLE_ROUTES_API_KEY", "")
ROUTE_MATRIX_URL = "https://routes.googleapis.com/distanceMatrix/v2:computeRouteMatrix"
ELEMENT_BUDGET = 12_000  # 10k free + $10 at $5/1000
CLUSTER_RADIUS_MILES = 0.5
TOP_K_CANDIDATES = 3

assert GOOGLE_API_KEY, (
    "Set GOOGLE_ROUTES_API_KEY in .env before running."
)
print(f"API key loaded: {GOOGLE_API_KEY[:8]}...")

API key loaded: AIzaSyAs...


## 1. Pharmacy Locations

Community/retail pharmacies in and near the four target counties, sourced from the NPPES NPI registry and cross-referenced with pharmacygps.com. Coordinates are pre-geocoded. We include pharmacies outside the target counties so that border-area addresses correctly find the nearest pharmacy even across county lines.

In [ ]:
# Fallback: hardcoded pharmacy data from NPPES/pharmacygps.com research,
# used if the live NPPES API is unavailable.
_FALLBACK_PHARMACIES = [
    # --- Caledonia County ---
    {"name": "Walgreens", "city": "St Johnsbury", "state": "VT", "address_1": "502 Railroad St", "postal_code": "05819", "lat": 44.4191, "lon": -71.9737},
    {"name": "Genoa Healthcare", "city": "St Johnsbury", "state": "VT", "address_1": "2225 Portland St", "postal_code": "05819", "lat": 44.4269, "lon": -71.9881},
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Lyndonville", "state": "VT", "address_1": "407 Broad St", "postal_code": "05851", "lat": 44.5260, "lon": -71.9977},
    {"name": "Lyndonville Pharmacy", "city": "Lyndonville", "state": "VT", "address_1": "101 Depot St", "postal_code": "05851", "lat": 44.5335, "lon": -72.0019},
    {"name": "Walgreens", "city": "Lyndonville", "state": "VT", "address_1": "412 Broad St", "postal_code": "05851", "lat": 44.5253, "lon": -72.0000},
    # --- Orleans County ---
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Newport", "state": "VT", "address_1": "55 Shattuck Hill Rd", "postal_code": "05855", "lat": 44.9398, "lon": -72.2057},
    {"name": "Walgreens", "city": "Newport", "state": "VT", "address_1": "59 Waterfront Plz", "postal_code": "05855", "lat": 44.9362, "lon": -72.2050},
    {"name": "Walgreens", "city": "Newport", "state": "VT", "address_1": "4408 US Route 5", "postal_code": "05855", "lat": 44.9524, "lon": -72.1559},
    {"name": "Genoa Healthcare", "city": "Newport", "state": "VT", "address_1": "181 Crawford Rd", "postal_code": "05855", "lat": 44.9350, "lon": -72.2100},
    {"name": "Brown's Drug Store", "city": "Derby Line", "state": "VT", "address_1": "40 Main St", "postal_code": "05830", "lat": 45.0053, "lon": -72.1001},
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Barton", "state": "VT", "address_1": "371 Barton Orleans Rd", "postal_code": "05822", "lat": 44.7508, "lon": -72.1804},
    # --- Lamoille County ---
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Morrisville", "state": "VT", "address_1": "97 Morrisville Plaza", "postal_code": "05661", "lat": 44.5617, "lon": -72.5959},
    {"name": "CVS Pharmacy", "city": "Morrisville", "state": "VT", "address_1": "13 VT Route 15 East", "postal_code": "05661", "lat": 44.5760, "lon": -72.5922},
    # Walgreens Morrisville (48 Congress St) — closed March 2023
    {"name": "Hannaford Pharmacy", "city": "Morrisville", "state": "VT", "address_1": "80 Fairgrounds Plz", "postal_code": "05661", "lat": 44.5590, "lon": -72.6010},
    {"name": "Genoa Healthcare", "city": "Morrisville", "state": "VT", "address_1": "72 Harrel St", "postal_code": "05661", "lat": 44.5630, "lon": -72.5980},
    {"name": "Lamoille Health Partners", "city": "Stowe", "state": "VT", "address_1": "1878 Mountain Rd", "postal_code": "05672", "lat": 44.4726, "lon": -72.6880},
    {"name": "Mountainside Pharmacy", "city": "Stowe", "state": "VT", "address_1": "45 Old Farm Rd", "postal_code": "05672", "lat": 44.4650, "lon": -72.6850},
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Cambridge", "state": "VT", "address_1": "27 North Main St", "postal_code": "05444", "lat": 44.6445, "lon": -72.8777},
    # --- Washington County ---
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Montpelier", "state": "VT", "address_1": "69 Main St", "postal_code": "05602", "lat": 44.2598, "lon": -72.5755},
    # Walgreens Montpelier (29 Main St) — closed November 2024
    {"name": "Barre Pharmacy", "city": "Barre", "state": "VT", "address_1": "20 S Main St", "postal_code": "05641", "lat": 44.1975, "lon": -72.5020},
    {"name": "CVS Pharmacy", "city": "Barre", "state": "VT", "address_1": "1634 US Route 302", "postal_code": "05641", "lat": 44.2105, "lon": -72.5330},
    {"name": "Walgreens", "city": "Barre", "state": "VT", "address_1": "355 N Main St", "postal_code": "05641", "lat": 44.2050, "lon": -72.5010},
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Barre", "state": "VT", "address_1": "800 US Rt 302", "postal_code": "05641", "lat": 44.2218, "lon": -72.5457},
    {"name": "Hannaford Pharmacy", "city": "Barre", "state": "VT", "address_1": "456 S Barre Rd", "postal_code": "05641", "lat": 44.1850, "lon": -72.5080},
    {"name": "Jo-Mar Inc", "city": "Barre", "state": "VT", "address_1": "921 US Rt 302", "postal_code": "05641", "lat": 44.2200, "lon": -72.5400},
    {"name": "Dog River Pharmacy", "city": "Northfield", "state": "VT", "address_1": "14 Depot Sq", "postal_code": "05663", "lat": 44.1488, "lon": -72.6564},
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Waterbury", "state": "VT", "address_1": "80 S Main St", "postal_code": "05676", "lat": 44.3376, "lon": -72.7564},
    {"name": "Shaw's Pharmacy", "city": "Waterbury", "state": "VT", "address_1": "820 Waterbury Stowe Rd", "postal_code": "05676", "lat": 44.3460, "lon": -72.7350},
    {"name": "Montpelier Pharmacy", "city": "Waterbury", "state": "VT", "address_1": "149 S Main St", "postal_code": "05676", "lat": 44.3350, "lon": -72.7570},
    # --- Nearby (outside target counties but reachable) ---
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Waitsfield", "state": "VT", "address_1": "Village Square", "postal_code": "05673", "lat": 44.1842, "lon": -72.8350},
    {"name": "Keene Medical Products", "city": "Montpelier", "state": "VT", "address_1": "81 River St", "postal_code": "05602", "lat": 44.2486, "lon": -72.5642},
    # Walgreens Hardwick (82 VT Route 15 W) — closed September 2024
    {"name": "KPH Healthcare (Kinney Drugs)", "city": "Island Pond", "state": "VT", "address_1": "92 Cross St", "postal_code": "05846", "lat": 44.8128, "lon": -71.8804},
]

print(f"Fallback list has {len(_FALLBACK_PHARMACIES)} pharmacies")

In [ ]:
pharmacies = pd.DataFrame(_FALLBACK_PHARMACIES)
pharmacies = pharmacies.dropna(subset=["lat", "lon"])
print(f"Loaded {len(pharmacies)} pharmacies")
pharmacies[["name", "city", "lat", "lon"]]

In [ ]:
# The closed Walgreens are already excluded from the curated list above
# (commented out with closure dates). This cell is a safety net for any
# future closures that need to be filtered from a cached or API-sourced list.
print(f"{len(pharmacies)} active pharmacies ready for analysis")

## 2. Cluster Nearby Pharmacies

Pharmacies within 0.5 miles of each other are grouped into a single centroid. This reduces destination count for the Route Matrix API, saving budget without losing meaningful geographic resolution.

In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3959  # Earth radius in miles
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2))
         * math.sin(dlon / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(a))


def cluster_pharmacies(df: pd.DataFrame, radius_miles: float) -> pd.DataFrame:
    assigned = set()
    clusters = []
    for i in df.index:
        if i in assigned:
            continue
        members = [i]
        assigned.add(i)
        for j in df.index:
            if j in assigned:
                continue
            d = haversine_miles(df.loc[i, "lat"], df.loc[i, "lon"],
                                df.loc[j, "lat"], df.loc[j, "lon"])
            if d <= radius_miles:
                members.append(j)
                assigned.add(j)
        subset = df.loc[members]
        clusters.append({
            "centroid_lat": subset["lat"].mean(),
            "centroid_lon": subset["lon"].mean(),
            "pharmacies": " / ".join(subset["name"]),
            "cities": " / ".join(subset["city"].unique()),
            "count": len(members),
        })
    return pd.DataFrame(clusters)


centroids = cluster_pharmacies(pharmacies, CLUSTER_RADIUS_MILES)
print(f"Clustered {len(pharmacies)} pharmacies into {len(centroids)} centroids")
print(f"  Single-pharmacy clusters: {(centroids['count'] == 1).sum()}")
print(f"  Multi-pharmacy clusters:  {(centroids['count'] > 1).sum()}")
centroids

## 3. County Boundaries

Download TIGER/Line county shapefiles from the US Census Bureau and filter to our four target counties.

In [ ]:
COUNTIES_CACHE = DATA_DIR / "counties.geojson"

TIGER_URL = (
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/"
    "cb_2023_us_county_500k.zip"
)


def fetch_county_boundaries() -> gpd.GeoDataFrame:
    """Download TIGER county shapefile and filter to target VT counties."""
    print("Downloading TIGER county shapefile...")
    counties_all = gpd.read_file(TIGER_URL)
    fips_codes = list(TARGET_COUNTIES.values())
    counties = counties_all[counties_all["GEOID"].isin(fips_codes)].copy()
    counties = counties.to_crs(epsg=4326)
    counties["county_name"] = counties["GEOID"].map(
        {v: k for k, v in TARGET_COUNTIES.items()}
    )
    return counties[["county_name", "GEOID", "geometry"]]


if COUNTIES_CACHE.exists():
    counties = gpd.read_file(COUNTIES_CACHE)
    print(f"Loaded {len(counties)} counties from cache")
else:
    counties = fetch_county_boundaries()
    counties.to_file(COUNTIES_CACHE, driver="GeoJSON")
    print(f"Cached {len(counties)} counties")

counties[["county_name", "GEOID"]]

## 4. Load Address Sample Points

Load the address CSVs for each county. These contain real addresses with geocoded coordinates, providing a realistic set of origins for drive-time computation.

In [ ]:
address_frames = []
for csv_path in sorted(ADDRESSES_DIR.glob("*.csv")):
    df = pd.read_csv(csv_path)
    county_fips = f"50{int(df['countyfp'].iloc[0]):03d}"
    df["county_name"] = COUNTY_FIPS_TO_NAME.get(county_fips, csv_path.stem)
    address_frames.append(df)

addresses = pd.concat(address_frames, ignore_index=True)
addresses = addresses.rename(columns={"latitude": "lat", "longitude": "lon"})
addresses = addresses.dropna(subset=["lat", "lon"])

print(f"Loaded {len(addresses)} addresses across {addresses['county_name'].nunique()} counties")
addresses.groupby("county_name").size()

## 5. Compute Drive Times via Google Routes API

**Strategy to stay within 12,000-element budget:**
1. Compute straight-line (Haversine) distance from every address to every centroid
2. Identify each address's top-3 closest centroids
3. **Phase 1**: Route Matrix for all addresses × nearest centroid (8,146 elements)
4. **Phase 2**: Remaining budget (~3,854 elements) for 2nd/3rd centroid checks on the most ambiguous addresses (where alternatives are within striking distance)
5. Take minimum drive time across all checked centroids per address

In [ ]:
DRIVE_TIMES_CACHE = DATA_DIR / "drive_times.csv"
BATCH_SIZE = 25  # max origins per Route Matrix request


def haversine_distances_to_centroids(addrs: pd.DataFrame, cents: pd.DataFrame) -> np.ndarray:
    """Return matrix of Haversine distances (miles): shape (n_addrs, n_centroids)."""
    R = 3959
    lat1 = np.radians(addrs["lat"].values[:, None])
    lon1 = np.radians(addrs["lon"].values[:, None])
    lat2 = np.radians(cents["centroid_lat"].values[None, :])
    lon2 = np.radians(cents["centroid_lon"].values[None, :])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def call_route_matrix(
    origins: list[tuple[float, float]],
    destinations: list[tuple[float, float]],
    api_key: str,
) -> list[dict]:
    """Call Google Routes API Compute Route Matrix. Returns list of elements."""
    body = {
        "origins": [
            {"waypoint": {"location": {"latLng": {"latitude": lat, "longitude": lon}}}}
            for lat, lon in origins
        ],
        "destinations": [
            {"waypoint": {"location": {"latLng": {"latitude": lat, "longitude": lon}}}}
            for lat, lon in destinations
        ],
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_UNAWARE",
    }
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": "originIndex,destinationIndex,duration,condition",
    }
    resp = requests.post(ROUTE_MATRIX_URL, json=body, headers=headers, timeout=120)
    resp.raise_for_status()
    return resp.json()


def compute_drive_times_adaptive(
    addrs: pd.DataFrame, cents: pd.DataFrame, budget: int, api_key: str
) -> pd.Series:
    """
    Compute min drive time (minutes) for each address using an adaptive
    budget strategy with the Google Routes API.
    """
    n_addrs = len(addrs)
    dist_matrix = haversine_distances_to_centroids(addrs, cents)
    top_k_indices = np.argsort(dist_matrix, axis=1)[:, :TOP_K_CANDIDATES]
    top_k_distances = np.take_along_axis(dist_matrix, top_k_indices, axis=1)

    drive_minutes = np.full(n_addrs, np.nan)
    elements_used = 0

    # --- Phase 1: every address × nearest centroid ---
    nearest_idx = top_k_indices[:, 0]
    unique_centroids = np.unique(nearest_idx)
    print(f"Phase 1: {n_addrs} addresses × nearest centroid ({len(unique_centroids)} unique)")

    for c_idx in unique_centroids:
        addr_mask = nearest_idx == c_idx
        addr_indices = np.where(addr_mask)[0]
        dest = [(cents.iloc[c_idx]["centroid_lat"], cents.iloc[c_idx]["centroid_lon"])]

        for batch_start in range(0, len(addr_indices), BATCH_SIZE):
            batch_idx = addr_indices[batch_start : batch_start + BATCH_SIZE]
            origins = [(addrs.iloc[i]["lat"], addrs.iloc[i]["lon"]) for i in batch_idx]
            elements_used += len(origins) * 1

            result = call_route_matrix(origins, dest, api_key)
            for elem in result:
                if elem.get("condition") == "ROUTE_NOT_FOUND":
                    continue
                dur_str = elem.get("duration", "0s")
                seconds = int(dur_str.rstrip("s"))
                oi = batch_idx[elem["originIndex"]]
                mins = seconds / 60.0
                if np.isnan(drive_minutes[oi]) or mins < drive_minutes[oi]:
                    drive_minutes[oi] = mins

            time.sleep(0.1)

        pct = elements_used / n_addrs * 100
        print(f"  elements: {elements_used}/{budget} ({pct:.0f}% of addresses done)")

    remaining = budget - elements_used
    print(f"\nPhase 1 complete: {elements_used} elements used, {remaining} remaining")

    # --- Phase 2: 2nd/3rd centroid for ambiguous addresses ---
    if remaining > 0 and TOP_K_CANDIDATES >= 2:
        ambiguity = top_k_distances[:, 1] / np.maximum(top_k_distances[:, 0], 0.1)
        ambiguous_idx = np.argsort(ambiguity)
        ambiguous_idx = ambiguous_idx[ambiguity[ambiguous_idx] < 2.0]

        extra_pairs = []
        for addr_i in ambiguous_idx:
            for k in range(1, TOP_K_CANDIDATES):
                if k < top_k_indices.shape[1]:
                    extra_pairs.append((addr_i, top_k_indices[addr_i, k]))
            if len(extra_pairs) >= remaining:
                break
        extra_pairs = extra_pairs[:remaining]
        print(f"Phase 2: {len(extra_pairs)} extra checks on ambiguous addresses")

        pairs_by_centroid: dict[int, list[int]] = {}
        for addr_i, c_idx in extra_pairs:
            pairs_by_centroid.setdefault(c_idx, []).append(addr_i)

        for c_idx, addr_indices_list in pairs_by_centroid.items():
            dest = [(cents.iloc[c_idx]["centroid_lat"], cents.iloc[c_idx]["centroid_lon"])]
            for batch_start in range(0, len(addr_indices_list), BATCH_SIZE):
                batch_idx = addr_indices_list[batch_start : batch_start + BATCH_SIZE]
                origins = [(addrs.iloc[i]["lat"], addrs.iloc[i]["lon"]) for i in batch_idx]
                elements_used += len(origins)

                result = call_route_matrix(origins, dest, api_key)
                for elem in result:
                    if elem.get("condition") == "ROUTE_NOT_FOUND":
                        continue
                    dur_str = elem.get("duration", "0s")
                    seconds = int(dur_str.rstrip("s"))
                    oi = batch_idx[elem["originIndex"]]
                    mins = seconds / 60.0
                    if np.isnan(drive_minutes[oi]) or mins < drive_minutes[oi]:
                        drive_minutes[oi] = mins

                time.sleep(0.1)

        print(f"Phase 2 complete: {elements_used} total elements used")

    cost = max(0, elements_used - 10_000) * 5 / 1000
    print(f"\nTotal: {elements_used} elements, estimated cost: ${cost:.2f}")
    return pd.Series(drive_minutes, index=addrs.index)


if DRIVE_TIMES_CACHE.exists():
    drive_times_df = pd.read_csv(DRIVE_TIMES_CACHE)
    print(f"Loaded {len(drive_times_df)} drive time records from cache")
else:
    print(f"Computing drive times: {len(addresses)} addresses, {len(centroids)} centroids")
    print(f"Budget: {ELEMENT_BUDGET} elements (${max(0, ELEMENT_BUDGET - 10000) * 5 / 1000:.0f} max cost)\n")
    min_times = compute_drive_times_adaptive(addresses, centroids, ELEMENT_BUDGET, GOOGLE_API_KEY)
    drive_times_df = addresses[["lat", "lon", "county_name", "city", "address_full"]].copy()
    drive_times_df["min_drive_minutes"] = min_times.values
    drive_times_df = drive_times_df.dropna(subset=["min_drive_minutes"])
    drive_times_df.to_csv(DRIVE_TIMES_CACHE, index=False)
    print(f"\nCached {len(drive_times_df)} drive time records")

drive_times_df.describe()

## 6. Heat Map Visualization

Interactive Folium map showing:
- **Color-coded circles** at each sample point representing drive time to nearest pharmacy
- **Pharmacy markers** showing each pharmacy location
- **County boundaries** as an overlay

Color scale: green (< 10 min) → yellow (10–20 min) → orange (20–30 min) → red (30+ min)

In [ ]:
import branca.colormap as cm


def drive_time_color(minutes: float) -> str:
    if minutes < 10:
        return "#2ecc71"   # green
    elif minutes < 15:
        return "#82e0aa"   # light green
    elif minutes < 20:
        return "#f9e79f"   # yellow
    elif minutes < 30:
        return "#f0b27a"   # orange
    elif minutes < 45:
        return "#e74c3c"   # red
    else:
        return "#8b0000"   # dark red


center_lat = drive_times_df["lat"].mean()
center_lon = drive_times_df["lon"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles="cartodbpositron")

# County boundaries
folium.GeoJson(
    counties.__geo_interface__,
    name="County Boundaries",
    style_function=lambda _: {
        "fillOpacity": 0.02,
        "color": "#333333",
        "weight": 2,
    },
    tooltip=folium.GeoJsonTooltip(fields=["county_name"], aliases=["County"]),
).add_to(m)

# Drive time circles
for _, row in drive_times_df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=4,
        color=drive_time_color(row["min_drive_minutes"]),
        fill=True,
        fill_color=drive_time_color(row["min_drive_minutes"]),
        fill_opacity=0.7,
        weight=0,
        tooltip=f"{row.get('address_full', '')}: {row['min_drive_minutes']:.1f} min".strip(": "),
    ).add_to(m)

# Pharmacy markers
for _, row in pharmacies.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color="#2c3e50",
        fill=True,
        fill_color="#3498db",
        fill_opacity=0.9,
        weight=1,
        tooltip=f"{row['name']} ({row['city']})",
    ).add_to(m)

# Legend
colormap = cm.LinearColormap(
    colors=["#2ecc71", "#82e0aa", "#f9e79f", "#f0b27a", "#e74c3c", "#8b0000"],
    index=[0, 10, 15, 20, 30, 45],
    vmin=0,
    vmax=60,
    caption="Drive time to nearest pharmacy (minutes)",
)
colormap.add_to(m)

folium.LayerControl().add_to(m)
m

## 7. Descriptive Statistics

Summary statistics, histograms, and box plots describing pharmacy access across the four counties.

In [ ]:
# Summary table: per-county and overall statistics
def compute_stats(df: pd.DataFrame, label: str) -> dict:
    d = df["min_drive_minutes"]
    total = len(d)
    return {
        "Region": label,
        "Sample Points": total,
        "Mean (min)": f"{d.mean():.1f}",
        "Median (min)": f"{d.median():.1f}",
        "Min (min)": f"{d.min():.1f}",
        "Max (min)": f"{d.max():.1f}",
        "P10 (min)": f"{d.quantile(0.10):.1f}",
        "P25 (min)": f"{d.quantile(0.25):.1f}",
        "P75 (min)": f"{d.quantile(0.75):.1f}",
        "P90 (min)": f"{d.quantile(0.90):.1f}",
        "% ≤ 10 min": f"{(d <= 10).sum() / total * 100:.1f}%",
        "% ≤ 15 min": f"{(d <= 15).sum() / total * 100:.1f}%",
        "% ≤ 20 min": f"{(d <= 20).sum() / total * 100:.1f}%",
        "% ≤ 30 min": f"{(d <= 30).sum() / total * 100:.1f}%",
    }


stats_rows = []
for county_name in sorted(TARGET_COUNTIES.keys()):
    subset = drive_times_df[drive_times_df["county_name"] == county_name]
    if len(subset) > 0:
        stats_rows.append(compute_stats(subset, county_name))

stats_rows.append(compute_stats(drive_times_df, "Overall"))

stats_table = pd.DataFrame(stats_rows).set_index("Region")
stats_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
for county_name in sorted(TARGET_COUNTIES.keys()):
    subset = drive_times_df[drive_times_df["county_name"] == county_name]
    ax.hist(
        subset["min_drive_minutes"],
        bins=30,
        alpha=0.5,
        label=county_name,
        edgecolor="white",
    )
ax.set_xlabel("Drive time to nearest pharmacy (minutes)")
ax.set_ylabel("Number of sample points")
ax.set_title("Distribution of Drive Times by County")
ax.legend()
ax.axvline(x=15, color="gray", linestyle="--", alpha=0.6, label="15 min")
ax.axvline(x=30, color="gray", linestyle=":", alpha=0.6, label="30 min")

# Box plot
ax = axes[1]
county_order = sorted(TARGET_COUNTIES.keys())
plot_data = [
    drive_times_df[drive_times_df["county_name"] == c]["min_drive_minutes"].values
    for c in county_order
]
bp = ax.boxplot(plot_data, labels=county_order, patch_artist=True)
colors = sns.color_palette("Set2", len(county_order))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
ax.set_ylabel("Drive time to nearest pharmacy (minutes)")
ax.set_title("Drive Time by County")
ax.axhline(y=15, color="gray", linestyle="--", alpha=0.6)
ax.axhline(y=30, color="gray", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig(DATA_DIR / "drive_time_stats.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to data/drive_time_stats.png")

In [ ]:
# CDF: cumulative percentage of area within X minutes of a pharmacy
fig, ax = plt.subplots(figsize=(10, 5))

for county_name in sorted(TARGET_COUNTIES.keys()):
    subset = drive_times_df[drive_times_df["county_name"] == county_name]["min_drive_minutes"].sort_values()
    cdf = np.arange(1, len(subset) + 1) / len(subset) * 100
    ax.plot(subset.values, cdf, label=county_name, linewidth=2)

overall = drive_times_df["min_drive_minutes"].sort_values()
cdf_all = np.arange(1, len(overall) + 1) / len(overall) * 100
ax.plot(overall.values, cdf_all, label="Overall", linewidth=2, color="black", linestyle="--")

ax.set_xlabel("Drive time to nearest pharmacy (minutes)")
ax.set_ylabel("Cumulative % of area")
ax.set_title("Cumulative Pharmacy Access by Drive Time")
ax.legend()
ax.set_xlim(0, max(60, overall.max()))
ax.set_ylim(0, 100)
ax.axvline(x=15, color="gray", linestyle="--", alpha=0.4)
ax.axvline(x=30, color="gray", linestyle=":", alpha=0.4)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / "drive_time_cdf.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to data/drive_time_cdf.png")

## Save Map to HTML

Export the interactive map as a standalone HTML file for sharing.

In [ ]:
map_path = DATA_DIR / "pharmacy_access_map.html"
m.save(str(map_path))
print(f"Interactive map saved to {map_path}")